# Week 5: индивидуальный чекпоинт — классификация аэрокосмических снимков (MLP, CNN и предобученный ResNet)

**Задача:** отнести каждый аэро- или спутниковый снимок (256×256 px) к одному из **20 классов сцен**
(бейсбольное поле, мост, церковь, облака, озеро, жилой квартал и т.д.).

Ноутбук основан на бейзлайне курса: все его разделы сохранены, к ним добавлены **анализ датасета** (раздел 3) и
**решения заданий** (разделы 10 и 12). Для запуска нужен архив соревнования с Kaggle; ссылка из бейзлайна
действует примерно неделю.

**Загрузка данных (один раз).** Ячейка ниже делает минимум работы:
1. данные уже распакованы в `/content` (например, после *Restart session*) → ничего не делает;
2. иначе берёт архив с Google Drive (`MyDrive/week-5-competition-geo-satellites-classification.zip`);
3. если на Drive его ещё нет — скачивает по ссылке курса и сохраняет копию на Drive на следующий раз.

Зачем так: архив весит ~300 МБ, а повторный `unzip` без флага `-o` останавливается с вопросом «заменить файлы?»,
и *Run all* на нём зависает. В каждой новой сессии Colab один раз попросит разрешение на доступ к Google Drive.

In [ ]:
import os
from pathlib import Path

ZIP_NAME = "week-5-competition-geo-satellites-classification.zip"
DRIVE_ZIP = Path("/content/drive/MyDrive") / ZIP_NAME

if Path("/content/train.csv").exists():
    print("data already unpacked")
else:
    if not Path(ZIP_NAME).exists():
        from google.colab import drive
        drive.mount("/content/drive")
        if DRIVE_ZIP.exists():
            print("copying the zip from Google Drive")
            !cp "{DRIVE_ZIP}" .
        else:
            !gdown 1sbnKeVmiOmT4ZZHIbXIxuBWF9CQLWt6s
            if Path(ZIP_NAME).exists():
                !cp {ZIP_NAME} "{DRIVE_ZIP}"
                print("saved a copy to Google Drive")
    !unzip -q -o {ZIP_NAME}
print(sorted(os.listdir("/content")))

**Данные**
```
train/  val/  test/          снимки, например train/1000.jpg
train.csv, val.csv           id,label     (id = имя файла, label = 0...19)
test.csv                     id           (снимки, которые нужно классифицировать)
classes.csv                  label,class_name
sample_submission_example    пример файла для Kaggle
```
Обучающая выборка маленькая (≈250 снимков на класс), тестовая — больше, как во многих реальных проектах, где
разметка дорогая. По условию validation и test взяты из одного распределения, и организаторы советуют
внимательно сравнить их с train — это мы делаем в разделе 3.

**План ноутбука**
1. Настройка
2. Чтение CSV-файлов
3. **Анализ датасета:** баланс классов, свойства снимков, чем val отличается от train
4. Преобразования и аугментации через `transforms.Compose`
5. `Dataset`, который читает снимки из папок
6. Функции обучения и оценки
7. Модель 1: MLP (с графиками loss и accuracy)
8. Модель 2: простая CNN (с графиками loss и accuracy)
9. Сравнение моделей
10. Своя CNN
11. **Задание:** предобученные модели и матрица ошибок (условие)
12. Решение: предобученный ResNet-18, эксперименты, сравнение, матрица ошибок, выводы
13. Предсказание test и создание `submission.csv`
14. Загрузка на Kaggle

**Метрика:** accuracy — доля правильно классифицированных тестовых снимков. Все классы одного размера, поэтому
accuracy учитывает их одинаково.

Запуск на GPU (Colab: *Среда выполнения → Сменить среду выполнения → T4 GPU*).

## 1. Настройка
`DATA_DIR` — папка, где лежат `train/`, `val/`, `test/` и CSV-файлы (в Colab это `/content/`, на Kaggle —
`/kaggle/input/<название соревнования>`).

**Воспроизводимость (правило соревнования).** `SEED` фиксирован, и `seed_everything()` вызывается перед
обучением каждой модели. Она задаёт seed для Python, NumPy и PyTorch и включает детерминированный режим cuDNN
(`deterministic = True`, `benchmark = False`): иначе GPU выбирает самые быстрые, но недетерминированные алгоритмы
свёртки, и результаты от запуска к запуску немного отличаются.

`QUICK_TEST = True` обучает каждую модель одну эпоху — только чтобы за несколько минут проверить, что ноутбук
доходит до конца без ошибок. Для настоящего результата `QUICK_TEST = False`.

In [ ]:
import io
import os
import copy
import time
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageEnhance, ImageFilter
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

DATA_DIR = Path(os.environ.get("CONTEST_DATA", "/content/"))
SEED = 42
QUICK_TEST = False   # True: по 1 эпохе на модель, только чтобы проверить, что ноутбук выполняется до конца
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything()
print("device:", DEVICE)
print("data folder:", sorted(p.name for p in DATA_DIR.iterdir()))

## 2. Чтение CSV-файлов
В таблицах только имена файлов и метки; сами снимки читаются позже. `dtype={"id": str}` нужен, чтобы id вида
`1000.jpg` гарантированно остались строками.

In [ ]:
train_table = pd.read_csv(DATA_DIR / "train.csv", dtype={"id": str})
val_table = pd.read_csv(DATA_DIR / "val.csv", dtype={"id": str})
test_table = pd.read_csv(DATA_DIR / "test.csv", dtype={"id": str})
classes = pd.read_csv(DATA_DIR / "classes.csv").sort_values("label").class_name.tolist()
NUM_CLASSES = len(classes)

print(f"train: {len(train_table)} images | val: {len(val_table)} | test: {len(test_table)} | classes: {NUM_CLASSES}")
print(classes)
train_table.head()

**Посмотрим на один обучающий снимок каждого класса.** Уже здесь видно, что классы различаются прежде всего
*текстурой и формой* объектов (ровные поля, сетка улиц, круглые развязки, рельсы), а не цветом. Поэтому дальше
важны модели, которые умеют находить локальные узоры, — свёрточные сети.

In [ ]:
def image_file(folder, image_id):
    """Путь к снимку. id — это имена файлов вида '1000.jpg' (просто числа тоже подходят)."""
    name = image_id if image_id.endswith(".jpg") else f"{image_id}.jpg"
    return DATA_DIR / folder / name

cols = (NUM_CLASSES + 1) // 2

fig, axes = plt.subplots(2, cols, figsize=(2 * cols, 4.6))
for label, ax in enumerate(axes.flat):
    ax.axis("off")
    if label < NUM_CLASSES:
        first_id = train_table[train_table.label == label].id.iloc[0]
        ax.imshow(Image.open(image_file("train", first_id)))
        ax.set_title(classes[label], fontsize=8)
plt.tight_layout(); plt.show()

## 3. Анализ датасета

Прежде чем выбирать модели, смотрим на сами данные. Нас интересуют три вопроса:
1. **Сколько снимков в каждом классе?** От этого зависит, честная ли метрика accuracy и нужны ли веса классов.
2. **Как снимки выглядят «в числах»** (размер, цвет, контраст, резкость)? От этого зависит предобработка.
3. **Отличаются ли val-снимки от train?** Test взят из того же распределения, что и val, поэтому всё, что мы
   найдём в val, ожидается и в test. Если val отличается от train, модель, обученная на train, будет ошибаться
   на test чаще, чем можно было бы ожидать.

Test-выборку мы **не анализируем**: по правилам test можно использовать только для получения предсказаний.
Val — законный «представитель» test, потому что организаторы прямо говорят, что распределение у них одно.

In [ ]:
counts = pd.DataFrame({"train": train_table.label.value_counts().sort_index(),
                       "val": val_table.label.value_counts().sort_index()})
counts.index = classes
print(counts.to_string())
print(f"\nimages per class: train {counts.train.min()}...{counts.train.max()}, val {counts.val.min()}...{counts.val.max()}")
print(f"test images: {len(test_table)} (labels hidden)")

**Классы сбалансированы:** 249–250 обучающих и 34–35 валидационных снимков в каждом классе. Что из этого следует:
* accuracy — честная метрика: каждый класс вносит в неё одинаковый вклад, модель не может «выиграть», угадывая
  только крупные классы;
* веса классов или пересэмплирование не нужны;
* модель, которая всегда отвечает одним классом, получила бы 1/20 = 5% — нижняя планка для всех моделей.

Дальше для каждого train- и val-снимка (в исходном размере) считаем несколько свойств:
* **размер** в пикселях;
* **средний цвет** (R, G, B);
* **резкость** — дисперсия лапласиана (второй производной яркости). Лапласиан велик там, где яркость резко
  меняется (края, мелкие детали). У размытого снимка резких перепадов почти нет, и дисперсия маленькая; шум,
  наоборот, добавляет «ложных» перепадов и увеличивает её;
* **контраст** — стандартное отклонение яркости;
* **насыщенность** — средняя насыщенность цвета; у чёрно-белого снимка она ≈ 0;
* **блочность** — во сколько раз перепады яркости на границах JPEG-блоков 8×8 сильнее, чем внутри блоков.
  JPEG сжимает каждый блок 8×8 отдельно, и при сильном сжатии на их границах появляются «ступеньки»; у
  нормального снимка это отношение ≈ 1.

In [ ]:
def image_stats(path):
    img = Image.open(path).convert("RGB")
    rgb = np.asarray(img, dtype=np.float32)
    gray = rgb @ np.array([0.299, 0.587, 0.114], dtype=np.float32)
    laplacian = gray[:-2, 1:-1] + gray[2:, 1:-1] + gray[1:-1, :-2] + gray[1:-1, 2:] - 4 * gray[1:-1, 1:-1]
    jumps = np.abs(np.diff(gray, axis=1))
    blockiness = jumps[:, 7::8].mean() / (np.delete(jumps, np.s_[7::8], axis=1).mean() + 1e-9)
    saturation = np.asarray(img.convert("HSV"), dtype=np.float32)[..., 1].mean()
    values = {"mean_r": rgb[..., 0].mean(), "mean_g": rgb[..., 1].mean(), "mean_b": rgb[..., 2].mean(),
              "sharpness": laplacian.var(), "contrast": gray.std(), "saturation": saturation, "blockiness": blockiness}
    return {"width": img.width, "height": img.height, **{k: float(v) for k, v in values.items()}}

rows = []
for split, table in [("train", train_table), ("val", val_table)]:
    for image_id, label in zip(tqdm(table.id, desc=f"stats {split}"), table.label):
        rows.append({"split": split, "id": image_id, "label": label, **image_stats(image_file(split, image_id))})
stats = pd.DataFrame(rows)

print("image sizes:", stats.groupby("split")[["width", "height"]].agg(["min", "max"]).to_string(), sep="\n")
print("\naverage values per split:")
print(stats.groupby("split")[["mean_r", "mean_g", "mean_b", "contrast", "saturation", "sharpness"]].mean().round(1).to_string())

**Как читать результат.** Все снимки 256×256 — приводить их к одному размеру не нужно. Средние значения у train
и val почти совпадают: цвет (≈ 90 / 93 / 85), контраст (35.9 и 35.1), резкость (965 и 959). Если остановиться
здесь, вывод был бы «выборки одинаковые».

Но среднее может скрывать **подмножество** необычных снимков: если часть снимков размыта, а остальные чуть резче
обычного, среднее почти не изменится. Поэтому смотрим на *хвосты* распределений.

Свойства зависят от класса (облака всегда гладкие, города всегда резкие), поэтому сравниваем **внутри каждого
класса**: для каждого класса берём 1-й и 99-й перцентили по *обучающим* снимкам и считаем, какая доля снимков
выходит за этот диапазон. Для train это ≈ 1% по построению. Если бы val был из того же распределения, у него
тоже получилось бы ≈ 1%. Сравнение внутри класса важно: иначе «резкость ниже нормы» могла бы означать просто
«в val больше облаков».

In [ ]:
train_stats = stats[stats.split == "train"]
low = train_stats.groupby("label")[["sharpness", "contrast"]].quantile(0.01).add_suffix("_q01")
high = train_stats.groupby("label")[["sharpness", "blockiness"]].quantile(0.99).add_suffix("_q99")
s = stats.join(low, on="label").join(high, on="label")

flags = pd.DataFrame({
    "blurry (sharpness < 1%)": s.sharpness < s.sharpness_q01,
    "noisy (sharpness > 99%)": s.sharpness > s.sharpness_q99,
    "strong JPEG (blockiness > 99%)": s.blockiness > s.blockiness_q99,
    "grayscale (saturation < 8)": s.saturation < 8,
    "low contrast (contrast < 1%)": s.contrast < s.contrast_q01,
})
flags["any of these"] = flags.any(axis=1)
print("Share of images (%) outside the 1%...99% range of the training images of the same class:")
print((flags.groupby(stats.split.to_numpy()).mean() * 100).round(1).T.to_string())

val_degraded = flags[stats.split.to_numpy() == "val"]["any of these"].to_numpy()   # в порядке строк val_table
print(f"\nvalidation images flagged as degraded: {val_degraded.sum()} of {len(val_degraded)}")

**Трактовка таблицы.**
* В train за диапазон выходит ≈ 1.2% снимков по каждому признаку — так и должно быть по построению (чуть больше
  1% из-за дискретности перцентилей), хотя бы по одному признаку — 4.4%.
* В val картина другая: **размытых в 11 раз больше** (13.0% против 1.2%), **сильно сжатых JPEG в 9 раз больше**
  (10.9%), **чёрно-белых в 15 раз больше** (3.0% против 0.2%), шумных и малоконтрастных — в 3–3.5 раза больше.
* Хотя бы один признак искажения есть у **27.2% val-снимков** (190 из 699) против 4.4% в train. Случайностью
  это быть не может: при доле 4.4% среди 699 снимков мы ждали бы около 31 ± 5 таких снимков, а их 190.
* Это **нижняя оценка**: порог в 1% ловит только сильные искажения, лёгкое размытие или сжатие сюда не
  попадают. Вероятно, искажена ещё большая доля val (и test).
* Маска `val_degraded` (190 снимков) понадобится позже: по ней мы посчитаем accuracy отдельно на чистых и на
  искажённых снимках и проверим, где именно ошибаются модели.

In [ ]:
# примеры val-снимков с найденными искажениями
val_flags = flags[stats.split.to_numpy() == "val"].reset_index(drop=True)
examples = [(column, i) for column in val_flags.columns[:-1] for i in np.flatnonzero(val_flags[column].to_numpy())[:2]]
fig, axes = plt.subplots(2, 5, figsize=(14, 6))
for ax, (column, i) in zip(axes.flat, examples):
    ax.imshow(Image.open(image_file("val", val_table.id.iloc[i])))
    ax.set_title(f"{column.split(' (')[0]}\n{classes[val_table.label.iloc[i]]}", fontsize=9)
for ax in axes.flat: ax.axis("off")
plt.tight_layout(); plt.show()

**Выводы анализа и что они значат для решения.**
1. **Классы сбалансированы** → accuracy — честная метрика, веса классов не нужны, случайное угадывание дало бы 5%.
2. **Все снимки 256×256, средние цвет и контраст у train и val совпадают** → одна и та же предобработка подходит
   для всех выборок.
3. **Около четверти val-снимков искажены** (размытие или низкое разрешение, сильное JPEG-сжатие, шум, ч/б,
   низкий контраст), а в train таких единицы. На примерах выше видно, что это действительно испорченные снимки, а
   не особенности классов: размытое теннисное поле, чёрно-белая церковь, «блочный» JPEG.
   Это **сдвиг распределения** (distribution shift) между обучением и тестом. Модель учится на тех картинках,
   которые видит: если в обучении не было размытых снимков, её признаки (края, мелкие текстуры) на размытом снимке
   не срабатывают, и она ошибается. Значит, на test стоит ждать больше ошибок, чем на «чистых» снимках.
4. **Что мы с этим делаем.** Менять val и test нельзя (и не нужно), но можно показать модели такие искажения во
   время обучения: аугментация `RandomCorruption` (раздел 12) случайно портит часть обучающих снимков теми же
   способами. Тогда модель учится узнавать теннисный корт и на размытом, и на чёрно-белом снимке. Помогает ли
   это, проверяем экспериментом (эксперимент 1 против 2) и отдельно считаем accuracy на чистых и искажённых
   val-снимках.

## 4. Преобразования и аугментации через `transforms.Compose`

Обучающая выборка заметно меньше тестовой. Частично это компенсируют **аугментации** — случайные изменения
обучающих снимков, которые не меняют их класс.

`transforms.Compose` — это список преобразований; каждая картинка проходит их **по порядку**.

**Обучающие снимки** получают случайные аугментации, поэтому в каждой эпохе модель видит немного другую версию
каждого снимка и не может просто запомнить обучающую выборку (это регуляризация). Для аэроснимков осмыслены:
* **отражения и повороты на 90°**: у снимка сверху нет направления «вверх», повёрнутое поле — всё ещё поле.
  Повороты именно на 90° не создают пустых углов и не интерполируют пиксели, а вместе с отражениями дают 8
  вариантов каждого снимка;
* **изменения яркости и контраста** (×0.8…1.2): одно и то же место выглядит по-разному в разное время суток и с
  разных камер.

**Validation- и test-снимки никогда не аугментируются**: они получают только то же преобразование в тензор и ту
же нормализацию, чтобы модель оценивалась на снимках такими, какие они есть.

`ToTensor()` превращает картинку в тензор формы (3, H, W) со значениями 0…1; `Normalize(0.5, 0.5)` переводит
их примерно в −1…1. Входы с нулевым средним и сравнимым масштабом ускоряют и стабилизируют градиентный спуск.

`IMG_SIZE = 32` — маленькие картинки, чтобы MLP и простая CNN обучались быстро. Плата за это — потеря деталей:
при 256 → 32 один пиксель модели соответствует квадрату 8×8 пикселей исходного снимка.

In [ ]:
IMG_SIZE = 32                                  # маленькие картинки: быстро, и для MLP и простой CNN этого хватает
MEAN, STD = (0.5, 0.5, 0.5), (0.5, 0.5, 0.5)

train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),                          # отражение слева направо, у половины снимков
    transforms.RandomVerticalFlip(p=0.5),                            # отражение сверху вниз, у половины снимков
    transforms.RandomChoice([transforms.RandomRotation((angle, angle)) # поворот на 0, 90, 180 или 270 градусов
                             for angle in (0, 90, 180, 270)]),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),            # яркость и контраст x0.8 ... x1.2
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

eval_transform = transforms.Compose([         # validation и test: для них никаких аугментаций!
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

print(train_transform)

**Что делают аугментации?** `Compose` — это просто список шагов, поэтому для превью можно взять все шаги, кроме
двух последних (`ToTensor` и `Normalize`), и показать один и тот же снимок шесть раз.

In [ ]:
preview_transform = transforms.Compose(train_transform.transforms[:-2])   # только аугментации

fig, axes = plt.subplots(2, 7, figsize=(14, 4.4))
for row, image_id in enumerate(train_table.id.iloc[:2]):
    img = Image.open(image_file("train", image_id)).convert("RGB").resize((IMG_SIZE, IMG_SIZE), Image.BOX)
    axes[row, 0].imshow(img); axes[row, 0].set_title("original", fontsize=9)
    for col in range(1, 7):
        axes[row, col].imshow(preview_transform(img)); axes[row, col].set_title("augmented", fontsize=9)
for ax in axes.flat: ax.axis("off")
plt.tight_layout(); plt.show()

## 5. `Dataset`, который читает снимки из папок
PyTorch `Dataset` отвечает на два вопроса: *сколько всего примеров?* (`__len__`) и *какой пример номер i?*
(`__getitem__`). Наш датасет читает id из CSV, загружает каждый снимок **один раз** (сразу уменьшая до
`IMG_SIZE` и храня в памяти — так обучение быстрее, чем читать JPEG с диска в каждой эпохе) и применяет
преобразования при каждом обращении, поэтому аугментации в каждой эпохе разные.

Уменьшение делается фильтром `Image.BOX`: он усредняет все исходные пиксели, попадающие в новый пиксель, и не
создаёт «ступенек» (алиасинга), как простое прореживание.

В test-CSV нет меток, для test-снимков датасет возвращает метку `-1`.

In [ ]:
class ContestDataset(Dataset):
    def __init__(self, table, folder, transform, img_size=IMG_SIZE):
        self.ids = table.id.tolist()
        self.labels = table.label.tolist() if "label" in table.columns else [-1] * len(table)
        self.transform = transform
        self.images = []
        for image_id in tqdm(self.ids, desc=f"loading {folder}"):
            with Image.open(image_file(folder, image_id)) as img:
                self.images.append(img.convert("RGB").resize((img_size, img_size), Image.BOX))

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, i):
        return self.transform(self.images[i]), self.labels[i]

train_set = ContestDataset(train_table, "train", train_transform)
val_set = ContestDataset(val_table, "val", eval_transform)
test_set = ContestDataset(test_table, "test", eval_transform)

image, label = train_set[0]
print("one sample:", tuple(image.shape), "| label:", label, f"({classes[label]})")

## 6. Функции обучения и оценки
`DataLoader` собирает примеры в батчи (и перемешивает обучающую выборку). Одни и те же функции используются для
всех моделей ноутбука, включая предобученные:

* `evaluate` считает loss и accuracy на выборке (без обучения и без аугментаций);
* `predict` возвращает предсказанный класс для каждого снимка;
* `train_model` обучает модель заданное число эпох. После каждой эпохи записывает **loss** и **accuracy** на
  train и val, а в конце возвращается к весам **лучшей эпохи** (с максимальной val accuracy);
* `make_loader` фиксирует порядок перемешивания генератором с `SEED`, чтобы результат воспроизводился.

Модели выдают «сырые» оценки классов (логиты); `nn.CrossEntropyLoss` сам применяет softmax и считает −log
вероятности правильного класса. Оптимизатор Adam — градиентный спуск, который для каждого веса подбирает свой
шаг по истории градиентов, поэтому хорошо работает без долгой настройки learning rate.

**Что изменено по сравнению с бейзлайном** (на сами модели это не влияет):
* `NUM_WORKERS = 2` — аугментации считаются в двух фоновых процессах, пока GPU обучает модель на предыдущем
  батче; иначе GPU простаивал бы, ожидая CPU. `seed_worker` задаёт seed каждому процессу, чтобы аугментации
  оставались воспроизводимыми.
* `QUICK_TEST` — режим быстрой проверки (одна эпоха).

**Замечание о выборе лучшей эпохи.** Эпоха выбирается по val, и по тому же val потом сообщается accuracy, поэтому
эта оценка немного **оптимистична**: из 12–30 значений, которые колеблются от шума, мы берём максимум. Разницу
между моделями в 1–2 п.п. не стоит считать надёжной.

In [ ]:
BATCH_SIZE = 64
criterion = nn.CrossEntropyLoss()

NUM_WORKERS = 2                                         # аугментации считаются в 2 фоновых процессах

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32           # берётся из генератора DataLoader -> воспроизводимо
    np.random.seed(worker_seed)
    random.seed(worker_seed)

def make_loader(dataset, shuffle):
    generator = torch.Generator().manual_seed(SEED)     # фиксированный порядок перемешивания -> воспроизводимо
    return DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=shuffle, generator=generator,
                      num_workers=NUM_WORKERS, worker_init_fn=seed_worker, pin_memory=(DEVICE == "cuda"))

@torch.no_grad()
def evaluate(model, dataset):
    model.eval()
    total_loss, correct = 0.0, 0
    for images, labels in make_loader(dataset, shuffle=False):
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        outputs = model(images)
        total_loss += criterion(outputs, labels).item() * len(images)
        correct += (outputs.argmax(dim=1) == labels).sum().item()
    return total_loss / len(dataset), correct / len(dataset)

@torch.no_grad()
def predict(model, dataset):
    model.eval()
    predictions = []
    for images, _ in make_loader(dataset, shuffle=False):
        outputs = model(images.to(DEVICE))
        predictions.append(outputs.argmax(dim=1).cpu())
    return torch.cat(predictions).numpy()

def train_model(model, name, epochs=30, lr=1e-3):
    if QUICK_TEST:
        epochs = 1
    seed_everything()
    model = model.to(DEVICE)
    trainable = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.Adam(trainable, lr=lr)
    train_loader = make_loader(train_set, shuffle=True)

    history, best_acc, best_weights = [], -1.0, None    # -1.0 — просто стартовое значение для сравнения
    for epoch in range(1, epochs + 1):

        # шаг обучения (такой же, как у более простых моделей раньше)
        model.train()
        total_loss, correct = 0.0, 0
        for images, labels in train_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * len(images)
            correct += (outputs.argmax(dim=1) == labels).sum().item()
        train_loss, train_acc = total_loss / len(train_set), correct / len(train_set)

        # шаг валидации
        val_loss, val_acc = evaluate(model, val_set)
        history.append({"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss,
                        "train_acc": train_acc, "val_acc": val_acc})

        if val_acc > best_acc:                                 # запоминаем лучшую эпоху и её веса
            best_acc, best_weights = val_acc, copy.deepcopy(model.state_dict())
        if epoch == 1 or epoch % 5 == 0:
            print(f"{name} | epoch {epoch:2d} | train loss {train_loss:.3f} | val loss {val_loss:.3f} | "
                  f"train acc {train_acc:.3f} | val acc {val_acc:.3f}")

    model.load_state_dict(best_weights)                        # возвращаемся к весам лучшей эпохи
    print(f"{name}: best validation accuracy {best_acc:.3f}")
    return pd.DataFrame(history)

def plot_history(history, name):
    """Loss (ошибка) и accuracy по эпохам для обучающей и валидационной выборок."""
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(history.epoch, history.train_loss, label="training")
    axes[0].plot(history.epoch, history.val_loss, label="validation")
    axes[0].set_title(f"{name}: loss (error)"); axes[0].set_xlabel("epoch"); axes[0].legend()
    axes[1].plot(history.epoch, history.train_acc, label="training")
    axes[1].plot(history.epoch, history.val_acc, label="validation")
    axes[1].set_title(f"{name}: accuracy"); axes[1].set_xlabel("epoch"); axes[1].legend()
    plt.tight_layout(); plt.show()

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

## 7. Модель 1: MLP

Построим простой многослойный перцептрон (MLP). Для этого используем:
- `nn.Sequential()` — упорядоченный контейнер PyTorch: модули идут цепочкой, и выход каждого автоматически
  становится входом следующего;
- `nn.Linear` — полносвязный слой: линейное преобразование `y = Wx + b` входного вектора;
- `nn.Flatten()` превращает каждый снимок 3×32×32 в вектор из 3 072 чисел; за ним идут два скрытых слоя (512 и
  256 нейронов) с ReLU и Dropout.

MLP считает каждый пиксель независимым числом: в отличие от свёртки, он не знает, какие пиксели соседние. Узор,
сдвинутый на один пиксель, для MLP — совсем другой вход. Поэтому от него ждём слабого результата: это нижняя
планка «нейросеть без знаний о структуре изображения».

In [ ]:
seed_everything()
mlp = nn.Sequential(
    nn.Flatten(),                                           # (3, 32, 32) -> 3072
    nn.Linear(3 * IMG_SIZE * IMG_SIZE, 512), nn.ReLU(), nn.Dropout(0.2),
    nn.Linear(512, 256), nn.ReLU(), nn.Dropout(0.2),
    nn.Linear(256, NUM_CLASSES),                            # по одной оценке (логиту) на класс
)
print(mlp)
print("trainable parameters:", count_parameters(mlp))

In [ ]:
history_mlp = train_model(mlp, "MLP")

**Loss и accuracy MLP.** Обучающие снимки аугментируются, а val — нет, поэтому кривые напрямую не сравнимы. Но если
train loss продолжает падать, а val loss растёт, модель начинает запоминать обучающие снимки (переобучение).

In [ ]:
plot_history(history_mlp, "MLP")

**Трактовка (MLP).** Лучшая val accuracy 0.373 — в 7 раз лучше случайного угадывания (0.05), но далеко от
пригодного. Важно, что и train accuracy низкая (≈ 0.32 на 30-й эпохе): модель **не может выучить даже обучающую
выборку**. Это недообучение (high bias): дело не в нехватке данных, а в том, что модель не подходит к задаче — она
не видит пространственных узоров, а на 32×32 к тому же пропали мелкие детали. Из 1.7 млн параметров MLP почти все
сидят в первом слое, который связывает каждый пиксель с каждым нейроном.

## 8. Модель 2: простая CNN

Теперь построим маленькую свёрточную сеть:
* `nn.Conv2d` — один слой свёрток 3×3 (32 фильтра). Каждый фильтр смотрит на маленькую окрестность пикселей и
  ищет в ней локальный узор (край, кусок крыши, дороги, границы поля) — **в любом месте снимка**, потому что один и
  тот же фильтр применяется ко всем позициям (разделение весов);
* `MaxPool2d(2)` уменьшает карту признаков вдвое: 32 → 16. Зачем: оставляет самый сильный отклик в каждом окне
  2×2, так что небольшой сдвиг объекта не меняет результат, и вчетверо сокращает число входов следующего слоя;
* классификатор на `nn.Linear` принимает решение по найденным признакам.

In [ ]:
seed_everything()
cnn = nn.Sequential(
    # извлечение признаков
    nn.Conv2d(3, 32, kernel_size=3, padding=1), nn.ReLU(),   # (3, 32, 32)  -> (32, 32, 32)
    nn.MaxPool2d(2),                                         # (32, 32, 32) -> (32, 16, 16)
    # классификация
    nn.Flatten(),                                            # (32, 16, 16) -> 32 * 16 * 16 = 8192
    nn.Linear(32 * (IMG_SIZE // 2) ** 2, 128), nn.ReLU(),
    nn.Linear(128, NUM_CLASSES),
)
print(cnn)
print("trainable parameters:", count_parameters(cnn))

In [ ]:
history_cnn = train_model(cnn, "CNN")

**Loss и accuracy CNN.** Сравните с графиками MLP выше.

In [ ]:
plot_history(history_cnn, "CNN")

**Трактовка (простая CNN).** Val accuracy 0.544 против 0.373 у MLP, хотя параметров меньше (1.05 млн против
1.7 млн). Свёртка — правильное **индуктивное смещение** для картинок: она изначально «знает», что важны локальные
узоры и что они могут быть где угодно на снимке. После ~20-й эпохи val accuracy почти не растёт (0.52–0.54), а
train продолжает расти (до 0.595), и val loss перестаёт падать (≈ 1.47) — модель начинает переобучаться.
Причина — большой полносвязный слой (8 192 × 128 ≈ 1 млн весов): он запоминает, *где* на снимке был узор. А
одного свёрточного слоя мало, чтобы собрать из краёв сложные объекты.

## 9. Сравнение моделей
У CNN **меньше параметров**, чем у MLP, но работает она заметно лучше — даже с одним свёрточным слоем. Значит,
важно не количество параметров, а то, насколько архитектура соответствует структуре данных.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for history, name in [(history_mlp, "MLP"), (history_cnn, "CNN")]:
    axes[0].plot(history.epoch, history.val_loss, label=name)
    axes[1].plot(history.epoch, history.val_acc, label=name)
axes[0].set_title("validation loss"); axes[1].set_title("validation accuracy")
for ax in axes: ax.set_xlabel("epoch"); ax.legend()
plt.tight_layout(); plt.show()

results = pd.DataFrame({
    "model": ["MLP", "CNN"],
    "parameters": [count_parameters(mlp), count_parameters(cnn)],
    "best val accuracy": [history_mlp.val_acc.max(), history_cnn.val_acc.max()],
})
print(results.round(3).to_string(index=False))

## 10. Своя CNN

Задание бейзлайна: поэкспериментировать с разными комбинациями и спроектировать свою CNN, используя такие
техники, как
- [Dropout](https://docs.pytorch.org/docs/2.14/generated/torch.nn.modules.dropout.Dropout.html#dropout$0)
- [Batch normalization](https://docs.pytorch.org/docs/2.14/generated/torch.nn.BatchNorm2d.html)

и добавить больше свёрточных слоёв: так модель может строить всё более сложные признаки.

### Наша CNN

Каждое решение — ответ на проблему, которую мы увидели у простой CNN:
* **Вход 64×64 вместо 32×32**: в 4 раза больше пикселей, мелкие объекты (корты, машины, рельсы) остаются
  различимы. Вычислений больше, но сеть всё равно обучается за пару минут.
* **4 блока [Conv 3×3 → BatchNorm → ReLU] × 2 → MaxPool.** Две свёртки 3×3 подряд видят область 5×5, но у них
  меньше весов, чем у одной свёртки 5×5 (2·9 = 18 против 25 на пару каналов), и между ними есть ещё одна
  нелинейность. Каждый блок уменьшает картинку вдвое (64 → 32 → 16 → 8 → 4) и удваивает число каналов
  (32 → 64 → 128 → 256): глубокие слои видят всё большую часть снимка и собирают из простых узоров (края)
  сложные (крыши, корты, развязки). Именно этого не хватало простой CNN с одним слоем.
* **BatchNorm** нормализует выходы каждого слоя по батчу (среднее 0, дисперсия 1, затем обучаемые масштаб и сдвиг).
  Масштаб сигналов не «плывёт» от слоя к слою, поэтому глубокую сеть можно учить с большим learning rate; шум от
  статистик батча работает как лёгкая регуляризация. `bias=False` у свёрток, потому что сдвиг всё равно добавит
  BatchNorm.
* **Global average pooling** вместо большого слоя `Flatten → Linear`: каждая из 256 карт признаков усредняется по
  всему снимку, и классификатор получает всего 256 чисел (≈ 5 тыс. весов вместо ≈ 1 млн). Это лечит главную
  проблему простой CNN: модели больше не нужно запоминать, *где* был узор, важно только, *есть ли* он. Для сцен
  это правильно: теннисный корт остаётся кортом в любом углу снимка.
* **Dropout 0.3** перед последним слоем случайно выключает 30% признаков во время обучения, чтобы классификатор не
  опирался на несколько признаков сразу, — против переобучения.
* **Аугментации и 30 эпох — те же**, что у MLP и простой CNN, чтобы сравнение было честным.

In [ ]:
CNN_SIZE = 64
cnn_train_transform = transforms.Compose(train_transform.transforms)   # те же аугментации, что и раньше
train_set = ContestDataset(train_table, "train", cnn_train_transform, img_size=CNN_SIZE)
val_set = ContestDataset(val_table, "val", eval_transform, img_size=CNN_SIZE)

def conv_block(c_in, c_out):
    return nn.Sequential(
        nn.Conv2d(c_in, c_out, kernel_size=3, padding=1, bias=False), nn.BatchNorm2d(c_out), nn.ReLU(inplace=True),
        nn.Conv2d(c_out, c_out, kernel_size=3, padding=1, bias=False), nn.BatchNorm2d(c_out), nn.ReLU(inplace=True),
        nn.MaxPool2d(2),
    )

seed_everything()
advanced_cnn = nn.Sequential(
    # извлечение признаков
    conv_block(3, 32),                     # (3, 64, 64)   -> (32, 32, 32)
    conv_block(32, 64),                    # (32, 32, 32)  -> (64, 16, 16)
    conv_block(64, 128),                   # (64, 16, 16)  -> (128, 8, 8)
    conv_block(128, 256),                  # (128, 8, 8)   -> (256, 4, 4)
    # классификация
    nn.AdaptiveAvgPool2d(1), nn.Flatten(), # (256, 4, 4)   -> 256
    nn.Dropout(0.3),
    nn.Linear(256, NUM_CLASSES),
)
print(advanced_cnn)
print("trainable parameters:", count_parameters(advanced_cnn))

In [ ]:
start = time.time()
history_adv = train_model(advanced_cnn, "Advanced CNN")
adv_minutes = (time.time() - start) / 60
print(f"training time: {adv_minutes:.1f} min")
plot_history(history_adv, "Advanced CNN")

**Трактовка (наша CNN).** Val accuracy 0.751 против 0.544 у простой CNN (+21 п.п.) при сопоставимом числе
параметров (1.18 млн). Вклад размера 64×64 и архитектуры здесь не разделён — менялись оба. По графикам:
* train и val accuracy растут вместе почти до конца; к 30-й эпохе train 0.82, val 0.74 — небольшой разрыв,
  умеренное переобучение;
* val-кривая «пилит» (±3–5 п.п. от эпохи к эпохе): в val всего 699 снимков, и BatchNorm ещё подстраивает свои
  статистики;
* к 30-й эпохе val accuracy ещё растёт — больше эпох, скорее всего, дали бы ещё немного.

Но сеть, обученная с нуля на 250 снимках на класс, упирается в объём данных: все признаки ей приходится выучивать
по этим 5 000 картинкам. Следующий шаг — взять признаки, выученные на миллионе снимков (разделы 11–12).

---
## 11. Задание: предобученные модели и матрица ошибок

Сети, обученные на ImageNet (1.2 млн фото, 1 000 классов), уже выучили общие визуальные признаки: края, текстуры,
формы. Их можно переиспользовать и обучить только небольшую часть под наши 20 классов.

**Разрешённые архитектуры** (никаких трансформеров и других семейств):

| Семейство | конструкторы torchvision | заменяемый последний слой |
|---|---|---|
| AlexNet | `alexnet` | `model.classifier` (последний слой: `model.classifier[6]`) |
| VGG | `vgg11`, `vgg13`, `vgg16`, `vgg19` (и версии `_bn`) | `model.classifier` (последний слой: `model.classifier[6]`) |
| Inception | `googlenet`, `inception_v3` | `model.fc` |
| ResNet | `resnet18`, `resnet34`, `resnet50`, `resnet101`, `resnet152` | `model.fc` |

### Пример: как подготовить предобученную модель
Пример ниже только **готовит** предобученный ResNet-18. Четыре шага:
1. загрузить модель с предобученными весами;
2. **заморозить** все её веса (`requires_grad = False`), чтобы обучение их не меняло;
3. **разморозить** последний блок (`requires_grad = True`), чтобы он мог подстроиться под аэроснимки;
4. убрать исходный классификатор на 1 000 классов и соединить сеть со **своим классификатором** через
   `nn.Sequential`.

In [ ]:
# 1) загружаем ResNet-18 с весами, предобученными на ImageNet
backbone = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
print(backbone)

In [ ]:
# 2) замораживаем все предобученные веса
for param in backbone.parameters():
    param.requires_grad = False

# 3) размораживаем последний residual-блок, чтобы его можно было дообучить
for param in backbone.layer4.parameters():
    param.requires_grad = True

# проверяем, какие части будут обучаться
for name, param in backbone.named_parameters():
    if param.requires_grad:
        print("trainable:", name)

In [ ]:
# 4) заменяем исходный классификатор своим и соединяем их через nn.Sequential
num_features = backbone.fc.in_features          # 512 у ResNet-18
backbone.fc = nn.Identity()                     # теперь backbone выдаёт 512 признаков на снимок

classifier = nn.Sequential(
    nn.Linear(num_features, 256),
    nn.LeakyReLU(),
    nn.Dropout(0.3),
    nn.Linear(256, NUM_CLASSES),                # без softmax: его применяет nn.CrossEntropyLoss
)

pretrained_model = nn.Sequential(backbone, classifier)

print(classifier)
print("trainable parameters:", count_parameters(pretrained_model),
      "of", sum(p.numel() for p in pretrained_model.parameters()))

# быстрая проверка: батч из 2 снимков 224x224 даёт 2 x 20 оценок классов
print("output shape:", tuple(pretrained_model(torch.randn(2, 3, 224, 224)).shape))

### Задачи
1. **Обучить предобученную модель** из разрешённого списка. Подготовить её как в примере (свой классификатор через
   `nn.Sequential`, разморожен хотя бы один предобученный слой или блок) и обучить через `train_model`.
2. **Адаптировать пайплайн данных.** Предобученные модели ждут более крупные картинки и нормализацию ImageNet:
   подумать об `IMG_SIZE` (например, 128 или 224; `inception_v3` нужно 299), `Normalize` со средним и std
   ImageNet и о том, какие аугментации помогают. Понадобятся новые пайплайны `transforms.Compose` и новые объекты
   `ContestDataset`.
3. **Сравнить** предобученную модель с MLP и своей CNN: таблица val accuracy и графики loss и accuracy
   (`plot_history`).
4. **Матрица ошибок** лучшей модели на val (строки — истинный класс, столбцы — предсказанный, с названиями классов
   на осях). Какие классы путаются чаще всего и почему? Посмотреть на несколько ошибочно классифицированных
   снимков. *Подсказка:* `predict(model, val_set)` даёт предсказания, `val_set.labels` — истинные классы; см.
   `sklearn.metrics.confusion_matrix` и `ConfusionMatrixDisplay`.
5. **Поэкспериментировать и объяснить:** какие слои размораживать, learning rate, аугментации, размер картинок.
   Какое изменение сильнее всего повлияло на val accuracy и почему?
6. **Отправить** лучшую модель на Kaggle с коротким описанием.

**Требования**
* Только AlexNet, VGG, Inception (GoogLeNet / Inception v3) и ResNet; никаких трансформеров и других архитектур.
* Предобученные веса — только из torchvision (ImageNet).
* Свои слои классификации, соединённые с предобученной сетью через `nn.Sequential`.
* Никаких внешних данных; ноутбук выполняется от начала до конца с фиксированным `SEED` и воспроизводит результат.

## 12. Решение: предобученный ResNet-18

**Идея переноса обучения (transfer learning).** ResNet-18 уже обучен на ImageNet. Его первые слои выучили
универсальные признаки — края, углы, текстуры; они полезны для любых изображений, включая аэроснимки. Последние
слои выучили признаки, специфичные для ImageNet (морды животных, колёса машин), их нужно перенастроить под наши
20 классов. Поэтому ранние слои мы **замораживаем** (они уже хороши, а 5 000 снимков скорее испортят их, чем
улучшат), а поздние **размораживаем** и дообучаем вместе со своим классификатором.

**Почему ResNet-18.** Из разрешённых семейств ResNet даёт лучшую точность при своём размере: residual-связи
(`выход = x + F(x)`) пропускают градиент через много слоёв напрямую, поэтому глубокие сети хорошо обучаются.
ResNet-18 — самая маленькая ResNet (11.7 млн параметров) и обучается за несколько минут на T4. VGG и AlexNet
гораздо больше (у VGG-16 — 138 млн параметров) и медленнее при меньшей точности на ImageNet; Inception v3 требует
вход 299×299.

**Задание 2: адаптация пайплайна данных.**
* `IMG_SIZE = 224` — размер, на котором обучались сети ImageNet: их фильтры настроены на объекты такого масштаба
  (эксперимент 5 проверяет, сколько стоит уменьшить до 128).
* `Normalize` со **средним и std ImageNet**: первый слой обучался на входах, нормализованных именно так; с другой
  нормализацией он получал бы сдвинутые входы.
* Аугментации: отражения, повороты на 90°, яркость и контраст, как раньше, и — как эксперимент —
  **RandomCorruption**, придуманная по итогам анализа датасета (раздел 3).

**Как устроена RandomCorruption.** Каждая операция соответствует одному из искажений, найденных в val:

| искажение в val (раздел 3) | операция | параметры |
|---|---|---|
| размытые (13%) | `blur` — гауссово размытие; `low_res` — уменьшение и обратное увеличение | sigma 0.8–3 px; в 2–6 раз |
| сильный JPEG (10.9%) | `jpeg` — пересохранение с низким качеством | качество 8–40 |
| шумные (4.1%) | `noise` — гауссов шум | std 5–20 уровней яркости |
| чёрно-белые (3%) | `gray` | — |
| низкий контраст (4%) | `low_contrast` | контраст ×0.4–0.8 |

С вероятностью p = 0.5 к снимку применяется одна (чаще) или две случайные операции. Почему 0.5: в val искажено не
меньше 27% снимков (это нижняя оценка), и при этом половина обучающих снимков остаётся чистой, чтобы модель не
разучилась работать с хорошими снимками. Параметры масштабируются под размер входа (`img_size / 256`), потому что
искажения измерялись на снимках 256×256. Портятся **только обучающие** снимки; val и test не меняются.

In [ ]:
IMAGENET_MEAN, IMAGENET_STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)

class RandomCorruption:
    # С вероятностью p применяет 1 или 2 случайных искажения, похожих на найденные в val-снимках.
    OPS = ("blur", "low_res", "jpeg", "gray", "noise", "low_contrast")

    def __init__(self, p=0.5, img_size=256):
        self.p = p
        self.scale = img_size / 256          # искажения измерялись на снимках 256x256

    def __call__(self, img):
        if random.random() >= self.p:
            return img
        for op in random.sample(self.OPS, k=random.choice([1, 1, 2])):
            img = getattr(self, op)(img)
        return img

    def blur(self, img):                     # гауссово размытие, sigma 0.8...3 px (для 256 px)
        return img.filter(ImageFilter.GaussianBlur(radius=random.uniform(0.8, 3.0) * self.scale))

    def low_res(self, img):                  # уменьшаем в 2...6 раз и увеличиваем обратно: мелкие детали теряются
        size = img.width
        small = max(8, int(size * random.uniform(1 / 6, 1 / 2)))
        return img.resize((small, small), Image.BILINEAR).resize((size, size), Image.BILINEAR)

    def jpeg(self, img):                     # сильное JPEG-сжатие, качество 8...40
        buffer = io.BytesIO()
        img.save(buffer, format="JPEG", quality=random.randint(8, 40))
        buffer.seek(0)
        return Image.open(buffer).convert("RGB")

    def gray(self, img):                     # оттенки серого, 3 канала сохраняются
        return img.convert("L").convert("RGB")

    def noise(self, img):                    # гауссов шум, std 5...20 уровней яркости
        rng = np.random.default_rng(random.getrandbits(32))
        pixels = np.asarray(img, dtype=np.float32)
        pixels = pixels + rng.normal(0, random.uniform(5, 20), pixels.shape)
        return Image.fromarray(np.clip(pixels, 0, 255).astype(np.uint8))

    def low_contrast(self, img):             # контраст x0.4...0.8
        return ImageEnhance.Contrast(img).enhance(random.uniform(0.4, 0.8))

def make_train_transform(img_size, corruption):
    steps = [
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.5),
        transforms.RandomChoice([transforms.RandomRotation((angle, angle)) for angle in (0, 90, 180, 270)]),
    ]
    if corruption:
        steps.append(RandomCorruption(p=0.5, img_size=img_size))
    steps += [transforms.ColorJitter(brightness=0.2, contrast=0.2),
              transforms.ToTensor(), transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)]
    return transforms.Compose(steps)

pretrained_eval_transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)])

In [ ]:
# превью: одни и те же train-снимки, чистые (слева) и после RandomCorruption(p=1)
random.seed(0)
preview_corruption = RandomCorruption(p=1.0)
fig, axes = plt.subplots(2, 7, figsize=(15, 4.6))
for row, image_id in enumerate(train_table.id.iloc[:2]):
    img = Image.open(image_file("train", image_id)).convert("RGB")
    axes[row, 0].imshow(img); axes[row, 0].set_title("original", fontsize=9)
    for col in range(1, 7):
        axes[row, col].imshow(preview_corruption(img)); axes[row, col].set_title("corrupted", fontsize=9)
for ax in axes.flat: ax.axis("off")
plt.tight_layout(); plt.show()
seed_everything()

**Превью.** Испорченные варианты одного и того же снимка похожи на настоящие искажённые val-снимки из раздела 3:
размытие, потеря деталей, серость, шум. Класс при этом человек по-прежнему узнаёт — значит, и модель может
научиться его узнавать.

**Задания 1 и 5: обучение и эксперименты.** Готовим ResNet-18 как в примере (предобученные веса, замороженные
слои, свой классификатор через `nn.Sequential`) и обучаем через `train_model` **только на `train.csv`**, 12 эпох,
с одним и тем же seed. 12 эпох вместо 30, потому что предобученная сеть сходится гораздо быстрее (уже ко 2–3-й эпохе
val accuracy выше 0.85).

Каждый эксперимент меняет **ровно одну** вещь по сравнению с предыдущим, поэтому эффект каждого изменения виден
отдельно и не смешивается с другими:

| # | вход | разморожено | аугментации | lr | вопрос |
|---|---|---|---|---|---|
| 1 | 224 | layer4 | базовые | 1e-3 | настройки из примера |
| 2 | 224 | layer4 | + искажения | 1e-3 | помогает ли аугментация с искажениями? (против 1) |
| 3 | 224 | layer3 + layer4 | + искажения | 1e-3 | помогает ли разморозить больше слоёв? (против 2) |
| 4 | 224 | layer3 + layer4 | + искажения | 3e-4 | лучше ли меньший learning rate? (против 3) |
| 5 | 128 | layer3 + layer4 | + искажения | 3e-4 | сколько стоят меньшие картинки? (против 4) |

Почему такие варианты:
* **layer4 или layer3 + layer4.** Чем глубже слой, тем специфичнее для ImageNet его признаки. Аэроснимки сильно
  отличаются от фото ImageNet, поэтому может быть полезно перенастроить и предпоследний блок. Но обучаемых весов
  становится больше (10.6 млн против 8.5 млн) — больше риск переобучиться на 5 000 снимках.
* **lr 1e-3 или 3e-4.** Предобученные веса уже близки к хорошему решению; слишком большой шаг может «сбить» их в
  первые эпохи, пока новый классификатор ещё случайный. Меньший шаг бережнее.
* **224 или 128 px.** На 128 px вычислений примерно в 3 раза меньше, но теряются детали.

**Насколько точны числа?** В val 699 снимков, поэтому стандартная ошибка accuracy около 0.90 равна
√(0.9·0.1/699) ≈ 0.011. Разница между моделями в 1–2 п.п. может быть случайной. Кроме того, `train_model`
выбирает эпоху с лучшей val accuracy, поэтому сообщаемая val accuracy немного завышена.

In [ ]:
EPOCHS_PRETRAINED = 12
pretrained_datasets = {}                     # img_size -> (train_set, val_set); снимки загружаются один раз для каждого размера
y_val = val_table.label.to_numpy()

def get_datasets(img_size):
    if img_size not in pretrained_datasets:
        pretrained_datasets[img_size] = (
            ContestDataset(train_table, "train", make_train_transform(img_size, corruption=False), img_size=img_size),
            ContestDataset(val_table, "val", pretrained_eval_transform, img_size=img_size),
        )
    return pretrained_datasets[img_size]

def make_resnet18(unfreeze):
    backbone = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    for param in backbone.parameters():                 # замораживаем все предобученные веса ...
        param.requires_grad = False
    for name in unfreeze:                               # ... и размораживаем выбранные блоки
        for param in getattr(backbone, name).parameters():
            param.requires_grad = True
    num_features = backbone.fc.in_features              # 512
    backbone.fc = nn.Identity()
    classifier = nn.Sequential(
        nn.Linear(num_features, 256), nn.LeakyReLU(), nn.Dropout(0.3),
        nn.Linear(256, NUM_CLASSES),
    )
    return nn.Sequential(backbone, classifier)

experiments, histories, val_predictions, trained_models = [], {}, {}, {}

def run_experiment(name, img_size, unfreeze, corruption, lr):
    global train_set, val_set                           # их использует train_model()
    train_set, val_set = get_datasets(img_size)
    train_set.transform = make_train_transform(img_size, corruption)
    seed_everything()
    model = make_resnet18(unfreeze)
    start = time.time()
    histories[name] = train_model(model, name, epochs=EPOCHS_PRETRAINED, lr=lr)
    minutes = (time.time() - start) / 60
    val_predictions[name] = predict(model, val_set)
    correct = val_predictions[name] == y_val
    experiments.append({
        "model": name, "img_size": img_size, "unfrozen": "+".join(unfreeze),
        "augmentation": "basic + corruption" if corruption else "basic", "lr": lr,
        "trainable params": count_parameters(model), "val accuracy": correct.mean(),
        "val acc (clean imgs)": correct[~val_degraded].mean(),
        "val acc (degraded imgs)": correct[val_degraded].mean(),
        "train minutes": round(minutes, 1), "seed": SEED,
    })
    trained_models[name] = model.cpu()
    torch.cuda.empty_cache()
    print(f"{name}: val accuracy {correct.mean():.4f} | {minutes:.1f} min\n")

In [ ]:
run_experiment("1 R18 224 layer4 basic lr1e-3",     224, ("layer4",),           corruption=False, lr=1e-3)
run_experiment("2 R18 224 layer4 corrupt lr1e-3",   224, ("layer4",),           corruption=True,  lr=1e-3)
run_experiment("3 R18 224 layer3+4 corrupt lr1e-3", 224, ("layer3", "layer4"), corruption=True,  lr=1e-3)
run_experiment("4 R18 224 layer3+4 corrupt lr3e-4", 224, ("layer3", "layer4"), corruption=True,  lr=3e-4)
run_experiment("5 R18 128 layer3+4 corrupt lr3e-4", 128, ("layer3", "layer4"), corruption=True,  lr=3e-4)

### Задание 3: сравнение всех моделей
В таблице: все модели, число обучаемых параметров, val accuracy с её стандартной ошибкой (± SE), время обучения
и для ResNet — accuracy отдельно на 509 чистых и 190 искажённых val-снимках (маска из раздела 3).

In [ ]:
n_val = len(y_val)
results_all = pd.DataFrame(
    [{"model": "MLP (32 px)", "trainable params": count_parameters(mlp), "val accuracy": history_mlp.val_acc.max()},
     {"model": "Simple CNN (32 px)", "trainable params": count_parameters(cnn), "val accuracy": history_cnn.val_acc.max()},
     {"model": "Advanced CNN (64 px)", "trainable params": count_parameters(advanced_cnn),
      "val accuracy": history_adv.val_acc.max(), "train minutes": round(adv_minutes, 1), "seed": SEED}]
    + experiments)
results_all["± SE"] = np.sqrt(results_all["val accuracy"] * (1 - results_all["val accuracy"]) / n_val)
pd.set_option("display.width", 250)
print(results_all.round(4).to_string(index=False))
results_all.to_csv("experiments.csv", index=False)

**Как читать таблицу.**
* **MLP → простая CNN → наша CNN → ResNet-18:** 0.373 → 0.544 → 0.751 → 0.904–0.934. Каждый шаг — более
  подходящее для картинок индуктивное смещение (свёртки, затем глубина и pooling, затем признаки с ImageNet), а не
  просто больше параметров.
* **Эксперимент 1 → 2 (аугментация с искажениями):** +2.4 п.п. в целом, но на искажённых снимках **+7.9 п.п.**
  (0.811 → 0.890), а на чистых почти ничего (+0.4). Это ровно тот механизм, который предсказал анализ данных:
  модель стала лучше там, где раньше не видела таких снимков. Заодно аугментация работает как регуляризация: в
  эксперименте 1 к 10-й эпохе train accuracy 0.95 при val 0.89, и val loss растёт (0.36 → 0.41 с 5-й по 10-ю
  эпоху) — переобучение; в эксперименте 2 такого разрыва нет, а val loss падает до 0.22.
* **Эксперимент 2 → 3 (разморозка layer3):** +0.4 п.п. — в пределах шума (SE ≈ 1 п.п.).
* **Эксперимент 3 → 4 (lr 3e-4):** +0.1 п.п. — тоже шум, но обучение заметно стабильнее (см. графики ниже), а на
  искажённых снимках 0.900 → 0.921.
* **Эксперимент 4 → 5 (128 px):** −2.0 п.п. (около двух SE), в основном на чистых снимках (0.939 → 0.916), зато
  обучение в 2.5 раза быстрее (1.7 против 4.2 мин).
* У ResNet ± SE ≈ 0.01, поэтому эксперименты 2, 3 и 4 статистически неразличимы. Уверенно можно говорить о вкладе
  предобучения и аугментации с искажениями.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
all_histories = [("MLP", history_mlp), ("Simple CNN", history_cnn), ("Advanced CNN", history_adv)] + list(histories.items())
for name, history in all_histories:
    axes[0].plot(history.epoch, history.val_loss, label=name)
    axes[1].plot(history.epoch, history.val_acc, label=name)
axes[0].set_title("validation loss"); axes[1].set_title("validation accuracy")
for ax in axes: ax.set_xlabel("epoch")
axes[1].legend(fontsize=7, loc="lower right")
plt.tight_layout(); plt.show()

best_name = max(experiments, key=lambda e: e["val accuracy"])["model"]
print("best pretrained model on validation:", best_name)
plot_history(histories[best_name], best_name)

**Трактовка графиков.**
* Все ResNet уже после 1–2 эпох лучше, чем наша CNN после 30: предобученные признаки полезны сразу.
* У эксперимента 3 (lr 1e-3, разморожен layer3) после 1-й эпохи val accuracy 0.694, у эксперимента 4 с lr 3e-4 —
  0.844: большой шаг в начале портит предобученные веса layer3, пока новый классификатор ещё случайный и даёт
  «шумные» градиенты. Потом сеть восстанавливается, но val loss на 10-й эпохе у неё вдвое выше (0.35 против 0.18).
* У лучшей модели (второй график) val accuracy большую часть времени **выше** train accuracy. Это не ошибка:
  обучающие снимки проходят через аугментации (половина испорчена) и Dropout, поэтому они «труднее» val. Train и
  val loss к концу сходятся (≈ 0.23) — переобучения нет.
* После 2-й эпохи val accuracy лучшей модели колеблется между 0.89 и 0.934; пик (0.934 на 10-й эпохе) и выбирает
  `train_model`. Этот разброс и есть шум оценки, о котором говорилось выше.

### Задание 4: матрица ошибок лучшей модели
Строки — истинные классы, столбцы — предсказанные. Каждая строка поделена на число снимков класса, поэтому на
диагонали — доля правильно классифицированных снимков каждого класса, а вне диагонали видно, куда «уходят» ошибки.
Строки нормированы, чтобы классы было легко сравнивать: в каждом классе 34–35 снимков, так что 0.03 ≈ один снимок.

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix

best_pred = val_predictions[best_name]
cm = confusion_matrix(y_val, best_pred, labels=range(NUM_CLASSES))
fig, ax = plt.subplots(figsize=(12, 11))
ConfusionMatrixDisplay(cm / cm.sum(axis=1, keepdims=True), display_labels=classes).plot(
    ax=ax, cmap="Blues", values_format=".2f", xticks_rotation=90, colorbar=False)
ax.set_title(f"{best_name}: validation confusion matrix (row-normalised)")
plt.tight_layout(); plt.show()

per_class = pd.Series(cm.diagonal() / cm.sum(axis=1), index=classes).sort_values()
print("accuracy per class (worst first):\n", per_class.round(3).to_string())

confusions = [(cm[t, p], classes[t], classes[p]) for t in range(NUM_CLASSES) for p in range(NUM_CLASSES)
              if t != p and cm[t, p] > 0]
print("\nmost frequent confusions (count, true -> predicted):")
for count, true_name, pred_name in sorted(confusions, reverse=True)[:10]:
    print(f"{count:3d}  {true_name} -> {pred_name}")

In [ ]:
# несколько ошибочно классифицированных val-снимков (исходные файлы 256x256)
wrong = np.flatnonzero(best_pred != y_val)
fig, axes = plt.subplots(3, 6, figsize=(15, 8.4))
for ax, i in zip(axes.flat, wrong[:18]):
    ax.imshow(Image.open(image_file("val", val_table.id.iloc[i])))
    ax.set_title(f"true: {classes[y_val[i]]}\npred: {classes[best_pred[i]]}" + ("\n(degraded)" if val_degraded[i] else ""),
                 fontsize=8)
for ax in axes.flat: ax.axis("off")
plt.tight_layout(); plt.show()
print(f"errors on degraded images: {val_degraded[wrong].sum()} of {len(wrong)} "
      f"(degraded images are {val_degraded.mean():.0%} of val)")

**Что видно на ошибках.** Около трети ошибочных снимков помечены как искажённые (сильное размытие,
JPEG-артефакты): на них пропадают детали, по которым классы различаются, — разметка корта, форма здания, граница
воды. Остальные ошибки — пары классов, похожих и для человека: часть вокзала без навесов выглядит как железная
дорога, жилой квартал с кортом во дворе — как теннисный корт. Подробно — в выводах ниже.

### Выводы

**Результаты** (val accuracy, 699 снимков, стандартная ошибка ≈ 0.01).

| модель | val accuracy | чистые снимки (509) | искажённые снимки (190) |
|---|---|---|---|
| MLP, 32 px | 0.373 | | |
| простая CNN, 32 px | 0.544 | | |
| наша CNN, 64 px | 0.751 | | |
| 1 ResNet-18, layer4, базовые аугментации, lr 1e-3 | 0.904 | 0.939 | 0.811 |
| 2 + аугментация с искажениями | 0.929 | 0.943 | 0.890 |
| 3 + разморожен layer3 | 0.933 | 0.945 | 0.900 |
| 4 + lr 3e-4 (**лучшая, по ней сабмит**) | **0.934** | 0.939 | 0.921 |
| 5 как 4, но 128 px | 0.914 | 0.916 | 0.911 |

**Какое изменение дало наибольший эффект и почему?**
* **Предобученные веса** — самый большой выигрыш: наша CNN, обученная с нуля, даёт 0.751, ResNet-18 с весами
  ImageNet — 0.904 (+15 п.п.). При 250 снимках на класс сеть не может выучить хорошие общие признаки (края,
  текстуры, формы) с нуля; ResNet-18 уже выучила их на 1.2 млн снимков, и мы лишь подстраиваем последние блоки
  под аэросъёмку.
* Среди экспериментов задания 5 сильнее всего сработала **аугментация с искажениями**: +2.4 п.п. в целом
  (≈ 2 стандартные ошибки), а val loss на 10-й эпохе упал с 0.41 до 0.22. Почти весь выигрыш — на искажённых
  снимках: 0.811 → 0.890 (+7.9 п.п.), на чистых почти без изменений (0.939 → 0.943). Ровно это предсказывал анализ
  данных: модель не видела размытых, чёрно-белых и пережатых снимков, а аугментация показывает их во время
  обучения.
* **Разморозка layer3** (+0.4 п.п.) и **меньший learning rate** (+0.1 п.п.) — в пределах шума. Но меньший lr
  делает обучение стабильнее: с lr 1e-3 val accuracy после первой эпохи упала до 0.694, потому что большие шаги
  портят предобученные веса layer3; с lr 3e-4 — 0.844, а val loss на 10-й эпохе вдвое меньше (0.18 против 0.35).
  Поэтому выбрана модель 4.
* **Меньшие снимки (128 вместо 224 px)** стоят 2 п.п. (в основном на чистых снимках, 0.939 → 0.916), зато обучение
  в 2.5 раза быстрее (1.7 против 4.2 мин): пропадают мелкие детали — машины, разметка кортов, рельсы, а
  предобученные фильтры рассчитаны на масштаб объектов ImageNet.
* Оговорки: по одному seed на эксперимент, и `train_model` берёт эпоху с лучшей val accuracy, поэтому все
  val-числа немного завышены. Разницу меньше ~2 п.п. не стоит переинтерпретировать.

**Какие классы путаются чаще всего и почему?**
* **church ↔ palace** и **church → commercial_area** (church — худший класс, 0.77): сверху все три — крупные здания
  со сложными крышами в плотной застройке; различает их *назначение* здания, а не внешний вид, и многие церкви
  стоят прямо в деловых кварталах.
* **railway ↔ railway_station**: на снимке вокзала есть пути, а часть вокзала без навесов над платформами выглядит
  в точности как железная дорога.
* **lake ↔ wetland**: и то и другое — вода с растительностью, разница в доле открытой воды. Размытое озеро (см.
  примеры ошибок) выглядит как болото.
* **terrace → rectangular_farmland**: оба класса — сельхозугодья с параллельными границами.
* Классы с уникальным цветом или текстурой (snowberg, sea_ice, cloud, runway, rectangular_farmland, overpass)
  распознаются без ошибок.
* Искажённые снимки — 27% val, но 33% ошибок (15 из 46): аугментация закрыла большую часть разрыва, но не весь.
  Некоторые ошибочные снимки размыты так сильно, что деталей, отличающих классы (разметка корта, форма здания),
  просто не осталось.

**Что можно сделать дальше** (за рамками чекпоинта): в командной фазе правила разрешают обучить финальную модель на
train+val; несколько seed помогли бы отделить эффекты экспериментов 2–4 от шума.

## 13. Предсказание test и создание `submission.csv`
Используем предобученную модель с лучшей val accuracy (эксперимент 4). Test-снимки получают ровно ту же
предобработку, что и val: уменьшение до 224 и нормализацию ImageNet, без аугментаций. Перед сохранением проверяем
формат: столбцы `id,label`, id в том же порядке, что в `test.csv`, без повторов, метки от 0 до 19 — ошибка формата
стоила бы попытки сабмита.

Почему не дообучаем на train+val: для индивидуального чекпоинта правила требуют обучать модели только на
`train.csv`, а val использовать для оценки и выбора модели.

In [ ]:
best_experiment = next(e for e in experiments if e["model"] == best_name)
test_set = ContestDataset(test_table, "test", pretrained_eval_transform, img_size=best_experiment["img_size"])
best_model = trained_models[best_name].to(DEVICE)

submission = pd.DataFrame({"id": test_set.ids, "label": predict(best_model, test_set)})

# проверка формата перед загрузкой
assert list(submission.columns) == ["id", "label"]
assert submission.id.tolist() == test_table.id.tolist()
assert submission.id.is_unique
assert submission.label.between(0, NUM_CLASSES - 1).all()

submission.to_csv("submission.csv", index=False)
print("model used:", best_name, "| rows:", len(submission))

**Результат на Kaggle.** Public leaderboard: **0.92444** (на val — 0.934). Разница около 1 п.п. ожидаема:
* val-оценка немного завышена, потому что лучшая эпоха выбиралась по тому же val;
* обе оценки шумные: SE на val ≈ 0.9 п.п., у public-части test — того же порядка.

То есть результат на test согласуется с val. На val эксперимент 1 (без аугментации с искажениями) был на 3 п.п.
хуже, поэтому без неё мы ожидали бы и на test результат ниже (это ожидание, на Kaggle оно не проверялось).

## 14. Загрузка на Kaggle
Скачайте `submission.csv` на компьютер (Colab: панель *Файлы* слева → правый клик → *Скачать*; ноутбук на
Kaggle: панель *Output*) и загрузите его на странице соревнования кнопкой *Submit Prediction*.

In [ ]:
import platform, sklearn, torchvision
print("python", platform.python_version(), "| torch", torch.__version__, "| torchvision", torchvision.__version__,
      "| numpy", np.__version__, "| pandas", pd.__version__, "| sklearn", sklearn.__version__,
      "| device", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")